In [ ]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional

from langchain_classic.prompts import load_prompt
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_community.vectorstores import FAISS

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

FAISS_PATH = "faiss_movies"
PROMPT_DIR = "prompts"

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print("벡터 수:", vectorstore.index.ntotal)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
class QueryParseResult(BaseModel):
    intent: str
    genres: List[str] = Field(default_factory=list)
    min_year: Optional[int] = None
    max_year: Optional[int] = None
    min_rating: Optional[float] = None
    max_runtime: Optional[int] = None
    mood: List[str] = Field(default_factory=list)
    audience: List[str] = Field(default_factory=list)
    similar_to: List[str] = Field(default_factory=list)
    keywords: List[str] = Field(default_factory=list)


class PreferenceProfile(BaseModel):
    taste_summary: str
    preferred_genres: List[str] = Field(default_factory=list)
    preferred_moods: List[str] = Field(default_factory=list)
    preferred_themes: List[str] = Field(default_factory=list)

In [ ]:
query_prompt = load_prompt(f"{PROMPT_DIR}/query_parsing.yaml")
preference_prompt = load_prompt(f"{PROMPT_DIR}/preference_analysis.yaml")
explanation_prompt = load_prompt(f"{PROMPT_DIR}/recommendation_explanation.yaml")
followup_prompt = load_prompt(f"{PROMPT_DIR}/followup_update.yaml")
final_response_prompt = load_prompt(f"{PROMPT_DIR}/final_response.yaml")

In [ ]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

query_parser = PydanticOutputParser(pydantic_object=QueryParseResult)
preference_parser = PydanticOutputParser(pydantic_object=PreferenceProfile)
text_parser = StrOutputParser()

# structured prompt에 format_instructions 주입 

In [ ]:
query_prompt = query_prompt.partial(
    format_instructions=query_parser.get_format_instructions()
)

preference_prompt = preference_prompt.partial(
    format_instructions=preference_parser.get_format_instructions()
)

followup_prompt = followup_prompt.partial(
    format_instructions=query_parser.get_format_instructions()
)

chain 구성

In [ ]:
query_parsing_chain = query_prompt | llm | query_parser
preference_analysis_chain = preference_prompt | llm | preference_parser
recommendation_explanation_chain = explanation_prompt | llm | text_parser
followup_update_chain = followup_prompt | llm | query_parser
final_response_chain = final_response_prompt | llm | text_parser

retrieval 함수

In [ ]:
def retrieve_movies(user_query, k=5):
    return vectorstore.similarity_search(user_query, k=k)

재정렬 함수

In [ ]:
def rerank_results(results):
    scored = []
    for doc in results:
        avg_rating = doc.metadata.get("avg_rating", 0) or 0
        rating_count = doc.metadata.get("rating_count", 0) or 0
        score = float(avg_rating) + min(float(rating_count) / 1000, 1.0)
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored]

Memory component 추가

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

E2E 추천 함수

In [ ]:
def build_recommendation_payload(user_query: str):
    # 1) 질의 구조화
    parsed_query = query_parsing_chain.invoke({
        "user_query": user_query
    })

    # 2) retrieval
    results = retrieve_movies(user_query, k=5)
    results = rerank_results(results)

    # 3) 각 영화별 추천 이유 생성
    movie_reasons = []
    for doc in results[:3]:
        reason = recommendation_explanation_chain.invoke({
            "user_query": user_query,
            "taste_profile": "",
            "movie_info": doc.page_content
        })
        movie_reasons.append({
            "title": doc.metadata.get("title"),
            "year": doc.metadata.get("year"),
            "reason": reason
        })

    return {
        "parsed_query": parsed_query,
        "movie_reasons": movie_reasons
    }

히스토리 반영 프롬프트

In [ ]:
final_chat_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 CineMate라는 영화 추천 도우미입니다. "
     "과거 대화 히스토리를 참고하여 사용자의 취향과 직전 요청을 반영하세요. "
     "입력에 없는 영화 정보는 만들어내지 마세요. "
     "항상 한국어로 응답하세요."),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "사용자 요청:\n{user_query}\n\n"
     "추천 결과:\n{recommended_movies_with_reasons}")
])

Memory-aware 최종 응답 chain

In [ ]:
memory_response_chain = final_chat_prompt | llm | text_parser

RunnableWithMessageHistory로 감싸기

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory

def invoke_cinemate_core(inputs: dict):
    user_query = inputs["user_query"]

    payload = build_recommendation_payload(user_query)

    return memory_response_chain.invoke({
        "history": inputs["history"],
        "user_query": user_query,
        "recommended_movies_with_reasons": json.dumps(
            payload["movie_reasons"],
            ensure_ascii=False,
            indent=2
        )
    })

cinemate_core = RunnableLambda(invoke_cinemate_core)

cinemate_with_memory = RunnableWithMessageHistory(
    cinemate_core,
    get_session_history,
    input_messages_key="user_query",
    history_messages_key="history",
)

End-to-End 실행

In [ ]:
session_id = "user-001"

response1 = cinemate_with_memory.invoke(
    {"user_query": "가족이랑 보기 좋은 2시간 이하 감동적인 영화 추천해줘"},
    config={"configurable": {"session_id": session_id}}
)

print(response1)

후속 질문

In [ ]:
response2 = cinemate_with_memory.invoke(
    {"user_query": "조금 더 가볍고 웃긴 영화로 바꿔줘"},
    config={"configurable": {"session_id": session_id}}
)

print(response2)